# What Makes a Good Shot? Expected Goals in Europe's Big Four Leagues, 2015/16
### DSE6311 Finalized Project Proposal

<img src="images/statsbomb_logo.png" width="210">

*Data provided by StatsBomb (StatsBomb Open Data, <https://github.com/hudl/open-data>).*

## Basics

| Item | Detail |
|---|---|
| **Project title** | What Makes a Good Shot? Expected Goals in Europe's Big Four Leagues, 2015/16 |
| **Team name** | Team Beta |
| **GitHub repository** | <https://github.com/aldengeorge26-droid/Soccer-Analytics-Capstone> |
| **Data source** | StatsBomb Open Data (credited with name and logo above) |

**Team members and standing roles**

| Member | Standing role |
|---|---|
| Brian Quevillon | Documentation|
| Alden George | Documentation |
| Brian D'Alelio | Communication |
| Anthony Dominici | ROLE |

**Weekly leads**

| Week | Roadmap milestone | Lead | Support |
|---|---|---|---|
| 4 | Polished EDA and unsupervised model | NAME | NAME |
| 5 | Baseline model | NAME | NAME |
| 6 | Additional models and tuning | NAME | NAME |
| 7 | Final model | NAME | NAME |
| 8 | Final report and presentation | NAME | NAME |

## Background & Project Question

Goals are rare in soccer, so a few deflections or good saves can change how a season looks. Clubs still judge teams on results, which can lead to bad decisions like firing a coach whose team creates good chances but fails to finish them (Brechot & Flepp, 2020). Expected goals (xG), the probability that a shot becomes a goal, separates chance creation from finishing and luck. xG predicts future success better than goals or shot totals (Mead et al., 2023), and Liverpool used similar analysis before hiring Jürgen Klopp (Schoenfeld, 2019). Distance and angle drive goal prediction (Rathke, 2017), and defender and goalkeeper positions improve location-only models (Lucey et al., 2015; Anzer & Bauer, 2021).

**Research question.** *Which characteristics of a shot (where it is taken, how the chance was created, and what the defense looks like at that moment) best predict its probability of becoming a goal?*

We aim for a well-calibrated xG model that shows coaching staff which chances are worth creating, without treating features as causes. This is hard because only about one in ten non-penalty shots is scored, the model must give realistic probabilities, and the same players and matches repeat, so splits must avoid leakage.

**Hypothesis.** A shot's chance of scoring is related to distance, angle, and where defenders and the goalkeeper are positioned, since these affect how much of the goal the shooter can aim at. How the chance was created may also matter. We expect that adding freeze-frame features, followed by build-up features, will lower log loss and Brier score compared with a location-only model when tested on later matches from the same season.

**Updates to the analysis plan.** We completed the chronological train/test split and set up separate leave-one-league-out splits using only the training matches. The test matches will be scored only once, after all tuning and model selection. We also compared three approaches to missing receipt values. Since their cross-validation results were nearly identical, we chose to replace missing values with zero and add a missing-value flag. We will compare feature groups in stages and calculate confidence intervals by sampling whole matches rather than individual shots, since shots from the same match are related.

## Data Preparation and Preprocessing

### Data Cleaning

**Scope and filtering.** The original StatsBomb Open Data contained 108,281 shots. We excluded shots from 274 matches without matching records and removed women's competitions, leaving 66,854 men's shots across 2,651 matches.

We checked which leagues had enough data for a full-season analysis. La Liga, the Premier League and Serie A each had all 380 matches from 2015/16, while Ligue 1 had 377. The other competitions were mostly tournaments, partial seasons or seasons covering one team. We kept these four leagues, giving us 37,888 shots across 1,517 matches. The three missing Ligue 1 matches make up only 0.2% of the expected total, so they should have little effect on our results.

We then removed 400 penalties because they are taken from a fixed position and have a much higher goal rate (75% versus 9.5%). We also removed 32 shots with no goalkeeper in the freeze frame. Of these, 19 were recorded as saves, showing that a goalkeeper was present but not captured. Seven direct corner shots were excluded because they were rare, with one recorded beyond the goal line. Finally, we removed 6 shots that had an assisting pass but no valid linked ball receipt. This left **37,443 shots with a 9.5% goal rate**.

**Missing values.** StatsBomb records True/False flags such as `under_pressure` and `shot_first_time` only when they are true, leaving them blank otherwise. We filled these missing values with False. About 28% of shots have missing receipt-to-shot values. Most had no assisting pass, while 15 had a valid receipt followed by another on-ball event before the shot. We kept these shots because their other features were still available and the missing values can reflect how the chance developed, such as a rebound or a shot after a dribble. The missing values are handled during preprocessing, as described in the Transformations section.

**Duplicates.** We checked for duplicate shots using StatsBomb's unique event `id`. No duplicate shot IDs or duplicate rows were found.

**Leakage fields.** We excluded from the model inputs any information that would not be available when the shot is taken, including the shot outcome, end location, deflections and final match score. The shot outcome was used to create our target, `is_goal`. StatsBomb's xG was kept only as a reference point, since its model may use different features and training data.

**Outliers.** We checked unusual shot distances, angles and defensive positions in `initial_eda_03`. No shots in the final dataset were recorded outside the pitch, and the calculated distances and angles matched their coordinates. Some extreme values represented real scoring opportunities. For example, 272 shots had the goalkeeper further from goal than the shooter, and 64% resulted in goals. We also kept 44 shots taken from the shooter's own half, none of which resulted in goals. These observations represent real game situations that our model should learn from.

### Transformations

**Encoding categorical variables.** Shot type, body part, technique and play pattern were one-hot encoded. Categories were learned from the training data only, so the test set could not influence which categories were included. True/False fields such as `under_pressure`, `shot_first_time` and `shot_one_on_one` were converted to 0 and 1.

**Scaling model inputs.** For logistic regression, we used `StandardScaler` to standardize the model inputs. The scaler is fitted on training data only and applied to the validation and test data using the same values.

**Collapsing categories.** Shot techniques appearing fewer than 150 times in the training data were grouped as `Rare or unseen`. This avoids creating separate predictors for techniques with very few shots and handles techniques not seen during training.

**Other transformations.** We calculated `shot_distance` and `shot_angle` using StatsBomb's 120 × 80 coordinate system. Both were checked against penalty shots and returned the expected 12 coordinate units and 36.87 degrees. We also used the freeze frames to count players in the shooting lane, estimate `percent_of_goal_open`, and measure goalkeeper positioning.

For shots following a pass, we calculated the time and straight-line distance between receiving the ball and shooting. Missing receipt measurements were filled with zero, with `receipt_measurements_missing` set to 1 to identify those shots. We compared this against median filling and dropping the features. All three performed nearly identically in cross-validation (log loss 0.2506–0.2507 and Brier score 0.0712), so we kept zero filling with the missing flag. Appendix A contains the full variable list.

The transformations that learn from data are fitted using training shots only and refitted within each cross-validation fold. This prevents validation data from influencing preprocessing. After encoding, the model inputs contain 37 predictors.

### Data Partitioning

**Training and test split.** We split the data by match date rather than randomly, holding out the newest 20% of matches for testing. Training contains 29,895 shots from 1,213 matches (August 7, 2015 to April 1, 2016), while testing contains 7,548 shots from 304 matches (April 2 to May 17, 2016). All shots from the same match stay together. The test set will only be used for final evaluation, not for feature selection, tuning or fitting a calibration model.

**Validation.** We used five-fold time-series cross-validation within the training set. Each fold trains on earlier matches and validates on later ones, with preprocessing fitted separately for each fold. This was used to select the regularization strength (C = 0.1, mean log loss 0.2507) and compare methods for handling missing receipt values.

**Why this approach fits our data.** Shots taken during the same match share teams, playing conditions and game state. If we split shots randomly, the same match could appear in both training and testing, making the model's performance look better than it really is. Splitting by date also reflects how a club would use the model, training on past matches to predict future ones. This allows us to test whether freeze-frame and build-up features improve predictions on later matches. The goal rate was 9.3% in training and 10.1% in testing, and each league accounted for 21% to 28% of test shots. We will check whether the higher test goal rate affects calibration.

**League generalization.** We also created leave-one-league-out splits using only the training matches. Each model will train on three leagues and be evaluated on the fourth to see how well it transfers. The chronological test set remains untouched.

**Uncertainty.** We will calculate confidence intervals by resampling whole matches rather than individual shots, since shots from the same match are related.

### Feature Engineering and Selection

**Created features.** We created three groups of features based on our research question.

**Shot location.** `shot_distance` and `shot_angle` describe where the shot is taken. Previous research identifies both as important predictors of scoring (Rathke, 2017), so they form our location-only baseline.

**Defensive positioning.** We used freeze frames to calculate `defenders_in_lane`, `teammates_in_lane`, `goalkeeper_in_lane`, `percent_of_goal_open`, `goalkeeper_distance_from_goal` and `goalkeeper_distance_to_shooter`. These describe where players and the goalkeeper are positioned when the shot is taken. Previous research found that these positions can improve location-only models (Lucey et al., 2015; Anzer & Bauer, 2021).

**Build-up.** We calculated the time and straight-line distance between receiving the ball and shooting. These features help describe how quickly a chance develops and how much the shooter moves before taking the shot. The `receipt_measurements_missing` flag identifies shots where these measurements could not be calculated.

We also kept StatsBomb's shot type, body part, technique, play pattern and situational flags, including `under_pressure` and `shot_first_time`.

**Feature selection.** We excluded post-shot information and kept StatsBomb's xG only as a reference point. After encoding, we have 37 predictors. Logistic regression uses L2 regularization to reduce the influence of less useful predictors.

**Planned comparisons.** Our primary goal is to outperform a constant-probability baseline on held-out data. StatsBomb's xG is shown only as a reference point, since its features and training data are unknown. We will compare four models in stages. Model A uses distance and angle, Model B adds freeze-frame features, Model C adds build-up features, and Model D adds shot context features. We will measure improvements using held-out log loss and Brier score. Feature importance will help explain the results but will not be used to test our hypothesis.

### Preprocessing Order

Both methods use the same 37,443 cleaned shots and chronological split. Any steps that learn from the data use training shots only. The test set is kept for final evaluation.

**Unsupervised methods (k-means clustering).** We will group shots using location, freeze-frame and build-up features, excluding `is_goal` and StatsBomb's xG. Missing receipt values will be filled with zero and flagged. We will use `StandardScaler` before clustering so features with larger values do not dominate the results.

The number of clusters will be chosen using silhouette scores or the elbow method on training data. We will then examine each cluster's characteristics and goal rate. Test shots will be assigned to the existing clusters without refitting. If we use cluster labels as model features, scaling and clustering will be repeated within each cross-validation fold.

**Supervised methods (logistic regression).** Within each cross-validation fold, we will group rare categories, one-hot encode categorical variables and scale the inputs using only the training shots. Boolean fields will be converted to 0 and 1, while missing receipt values will be filled with zero and flagged. The model will then be trained on earlier matches and evaluated on later validation matches.

We will compare four models. Model A uses distance and angle, Model B adds freeze-frame features, Model C adds build-up features, and Model D adds shot context features. We will also test smaller feature sets, such as using `percent_of_goal_open` instead of all six freeze-frame features. The best feature set and regularization strength will be chosen using mean validation log loss.

After selecting the model, we will refit it on the full training set and check calibration using cross-validated predictions. The test set will be scored once, with confidence intervals for each model and the differences between models calculated by resampling whole matches.

**Leave-one-league-out testing.** We will train on three leagues and test on the fourth, repeating this for all four leagues. Any tuning will use only the three training leagues. These results will be reported separately from the chronological test.

### Data Leakage Prevention

**Potential sources of leakage.** We identified six areas where information could affect our model results.

- **Post-shot information.** The shot outcome, where the ball ended up, deflections and final match score are only known after the shot. These cannot be used to predict whether the shot becomes a goal.
- **StatsBomb's xG.** This is already another model's prediction for the same shot, so we use it only as a reference point.
- **Time order.** Training on later matches and testing on earlier ones would allow the model to learn from games that had not happened yet.
- **Shots from the same match.** Shots from one match share teams, conditions and game state. Having the same match in both sets could make our model look better than it actually is.
- **Preprocessing and tuning.** Decisions about categories, scaling, clustering and model settings must come from training data. Using validation or test data for these decisions could influence our results.
- **Feature construction.** Receipt features use events leading up to a shot. We need to make sure they do not include anything that happened after the shot.

**Partitioning and validation design.** We split matches by date, keeping the newest 20% for final testing (see Data Partitioning). All shots from the same match stay together, and `pre_processing_04` confirms that no matches overlap between training and testing. Cross-validation also follows the time order, with preprocessing fitted separately within each fold. Leave-one-league-out testing uses only training matches, leaving the final test set untouched. Confidence intervals will be calculated by resampling whole matches rather than individual shots.

**Steps estimated from training data only.** Any preprocessing that learns from the data uses training shots only. During cross-validation, these steps are fitted separately within each fold.

- **Categories.** Category levels and rare shot techniques are determined from training shots, including which techniques have fewer than 150 examples.
- **Scaling.** `StandardScaler` calculates the mean and standard deviation of each feature using training data.
- **Feature selection and tuning.** We used cross-validation to choose the regularization strength (C) and will use it to compare feature sets.
- **Clustering.** The scaler, number of clusters and cluster centers are determined using training data.
- **Calibration.** We will use cross-validated predictions from the training set to check calibration and make any adjustments.

Missing receipt values are filled with zero, so no training statistics are needed. Features such as shot distance, angle and freeze-frame measurements are calculated directly from each shot. Receipt measurements use only events before the shot. These calculations do not depend on other shots in the training or test sets.

**Keeping validation and test data out of model development.** Validation folds are used to compare models, while all fitting uses the training shots within each fold. The `fit_shot_preprocessing` and `transform_shot_predictors` functions handle preprocessing, with `StandardScaler` included in the model pipeline.

The final test set will not be used until we have chosen the features, regularization strength, number of clusters and any calibration adjustments. We will record these decisions before testing the model once. If performance is worse than expected, we will report the results rather than adjust the model based on the test set. Any changes made afterward will be documented. For leave-one-league-out testing, tuning will use only the three training leagues.

**Other project-specific risks.** We addressed time and match-related leakage through the chronological split and by keeping shots from the same match together. Post-shot information and StatsBomb's xG were excluded from the model inputs. We found no separate spatial leakage concern because location and freeze-frame features describe each shot individually. Leave-one-league-out testing will help us see whether the model relies on patterns specific to certain leagues.

Our initial EDA used the full dataset, including matches from the test period, but these matches were not used for model fitting or tuning. Our features were chosen from previous research, our cleaning rules are based on individual shots, and the Isolation Forest outlier flags were used only for review, not for removing shots or as model inputs. We also excluded team and player identifiers to limit memorization. One limitation remains: the test set comes from the same season and leagues as the training data, so our results will show performance on later matches rather than a new season.